In [2]:
from lir_proteome_screen_pssm import environment as env
import pandas as pd
import lir_proteome_screen_pssm.sequence_utils as seqtools
import re
import numpy as np
import copy
import lir_proteome_screen_pssm.data_loaders as dl

old_regex = "...[FWY]..[LVI]"
new_regex = "...[FWY]..[WFY]"

# preprocessed sequence table from Jen

In [14]:
rename_dict = {
    "first_4_residues": "4mer",
    "first_6_residues": "6mer",
    "first_8_residues": "8mer",
    "first_14_residues": "14mer",
    "first_7_residues": "7mer",
    "first_5_residues": "5mer",
}

screen_df = pd.read_csv(env.RAWFILEPATHS.screening_hits_table)
screen_df = screen_df.rename(columns=rename_dict)
assert all(
    [len(s) == 7 for s in screen_df["7mer"].tolist()]
), "All binders should be 7 residues long"
assert all(
    screen_df["7mer"].str.contains('-') == False
), "No 7mers should contain gaps"

screen_binders_df = screen_df[screen_df["Bind/Nonbind"] == "Bind"].copy()
screen_binders_df["true label"] = 1
screen_binders_df = screen_binders_df.sort_values(by="avg_z_score", ascending=False)
screen_nonbinders_df = screen_df[screen_df['Bind/Nonbind'] == 'Nonbind'].copy()
screen_nonbinders_df["true label"] = 0
print(len(screen_binders_df))
a = set(screen_binders_df["7mer"].to_list())

297


In [15]:
screen_binders_df[['ID', 'Input Count', 'avg_z_score', '7mer']].to_csv("./09_binders_from_Jen.csv", index=False)

# from all data to preprocessed binders (a double check)

In [ ]:
full_data_table = pd.read_csv(env.RAWFILEPATHS.full_screening_table, sep='\t')
screen_binders_from_raw = full_data_table[full_data_table['avg_z_score'] >= 1.7].copy()
screen_binders_from_raw = screen_binders_from_raw[screen_binders_from_raw['Input Count'] >= 10].copy()
# regex extract 7mer from 7mer column
# only keep 

def get_regex_matches(s: pd.Series, regex: str):
    matches = list(seqtools.get_regex_matches(regex, s["ID"]))
    # if len(matches) == 0:
    #     return
    return matches

print(len(screen_binders_from_raw))
# REGEX = seqtools.regex2overlapping("...[FWY]..[ILVWFY]")
# REGEX = "...[FWY]..[ILVWFY]"
REGEX = "[FWY]..[ILVWFY]"
screen_binders_from_raw["regex_matches"] = screen_binders_from_raw.apply(get_regex_matches, axis=1, regex=REGEX)
screen_binders_from_raw["num_regex_matches"] = screen_binders_from_raw["regex_matches"].apply(lambda x: len(x))
screen_binders_from_raw["num_regex_matches"].value_counts()
df_multi = screen_binders_from_raw[screen_binders_from_raw["num_regex_matches"] > 1].copy()
df_multi = df_multi.explode("regex_matches")
df_single = screen_binders_from_raw[screen_binders_from_raw["num_regex_matches"] == 1].copy()
df_single["regex_matches"] = df_single["regex_matches"].apply(lambda x: x[0])
screen_binders_from_raw = pd.concat([df_multi, df_single])
screen_binders_from_raw[["7mer", "motif_start", "motif_end"]] = pd.DataFrame(
    screen_binders_from_raw["regex_matches"].tolist(), index=screen_binders_from_raw.index
)
screen_binders_from_raw = screen_binders_from_raw.sort_values(by="avg_z_score", ascending=False)
print(len(screen_binders_from_raw))
b = set(screen_binders_from_raw["7mer"].to_list())

420
307


In [19]:
screen_binders_from_raw[['ID', 'Input Count', 'avg_z_score', '7mer']].to_csv("./09_binders_from_Jackson.csv", index=False)

In [7]:
print(len(a), len(b))

283 260


In [8]:
screen_binders_from_raw[screen_binders_from_raw['7mer'].isin(b.difference(a))]

,Unnamed: 0,ID,ER 0,ER 1,ER 3,ER 4,ER 5,ER 6,Input Count,1 Count,...,Score (Bits),z_score_4,z_score_5,z_score_6,avg_z_score,regex_matches,num_regex_matches,7mer,motif_start,motif_end
5,5,IWYWSDEFGSWQEYGRQGTVTL,0,5.208992,9.572567,11.569644,12.334340,12.643501,319,6432,...,51.2,3.345774,3.710809,4.088600,3.72,"(SDEFGSW, 4, 10)",1,SDEFGSW,4,10
20,23,YLGQLEHEDIDTSADAVEDLTEAEWEDLTQQYYSLV,0,5.857058,9.449375,11.147782,11.561737,11.270682,124,3918,...,80.1,3.165840,3.389663,3.513537,3.36,"(TQQYYSL, 28, 34)",2,TQQYYSL,28,34
27,30,LDGTAVENIETFQTEDHTFDEYTEELDCWVVWE,0,5.473140,9.478454,10.950355,11.217866,11.141941,42,1017,...,53.1,3.081632,3.246726,3.459609,3.26,"(DHTFDEY, 15, 21)",2,DHTFDEY,15,21
27,30,LDGTAVENIETFQTEDHTFDEYTEELDCWVVWE,0,5.473140,9.478454,10.950355,11.217866,11.141941,42,1017,...,53.1,3.081632,3.246726,3.459609,3.26,"(LDCWVVW, 25, 31)",2,LDCWVVW,25,31
41,45,NRVTVYEYDTREDQWINIGTI,0,4.826754,8.687136,10.142549,10.645705,10.688468,200,3094,...,49.7,2.737083,3.008897,3.269653,3.01,"(EDQWINI, 11, 17)",1,EDQWINI,11,17
48,53,MFHKAEELFSKTTNNEVDDMDTSDTQWGWF,0,4.941442,8.601221,10.013964,10.283439,10.441986,68,1139,...,66.6,2.682238,2.858315,3.166403,2.90,"(DTQWGWF, 23, 29)",1,DTQWGWF,23,29
54,59,ESSTEWDLDSFSELDSESGSSSSFSDDEVWVQVAPQ,0,6.625375,9.298088,10.101788,10.390150,9.487462,11,592,...,70.1,2.719697,2.902671,2.766561,2.80,"(LDSFSEL, 7, 13)",2,LDSFSEL,7,13
66,71,VTPDSGYSSAHAEATYEEDWEVFDPYYFIKRPATD,0,5.932803,8.641368,9.730574,9.958704,9.744875,60,1998,...,67.4,2.561365,2.723333,2.874389,2.72,"(EEDWEVF, 16, 22)",1,EEDWEVF,16,22
89,94,IWYWSDEFGSWQEYGRQGTVHPVTTVSSSDVEKAYL,0,6.032177,8.032407,9.444030,9.577986,9.270664,169,6029,...,79.7,2.439147,2.565080,2.675746,2.56,"(SDEFGSW, 4, 10)",1,SDEFGSW,4,10
113,118,KEELVMSSLKRLNSYLSLPKFRSFRTY,0,5.230196,8.516249,9.457672,9.272525,8.409952,13,266,...,22.3,2.444966,2.438110,2.315201,2.40,"(FRSFRTY, 20, 26)",2,FRSFRTY,20,26


In [9]:
screen_binders_df[screen_binders_df['7mer'].isin(a.difference(b))]

,ID,Position,Hit,Length,Input Count,ER 0,ER 1,ER 3,4,5,...,24_mer,12_mer,sequence_length,nonLIR_6,nonLIR_8,nonLIR_7,nonLIR_5,7mer,5mer,true label
5,EDWDMLDVDEDEKLTGEEEFELLLVRLV,31,NaN,28,12,NaN,NaN,NaN,11.862116,12.408549,...,GGSGIPLREDWDMLDVDEDEKLTG,LREDWDMLDVDE,24,EDWDML,LREDWDML,REDWDML,DWDML,REDWDML,DWDML,1
13,DVWEILRRHRHLSTSASPSSTASLTCAAC,31,NaN,29,24,NaN,NaN,NaN,11.500042,12.169792,...,GGSGIPLRDVWEILRRHRHLSTSA,LRDVWEILRRHR,24,DVWEIL,LRDVWEIL,RDVWEIL,VWEIL,RDVWEIL,VWEIL,1
18,KLEVPTGPEVQTPKPSDADWDDLWTSLMGGGI,52,NaN,32,20,NaN,NaN,NaN,11.253341,11.691843,...,KPSDADWDDLWTSLMGGGI-----,WDDLWTSLMGGG,24,DLWTSL,WDDLWTSL,DDLWTSL,LWTSL,DDLWTSL,LWTSL,1
21,PYDSYNPSVLRGPLLGHTDAVWGLL,30,NaN,25,30,NaN,NaN,NaN,10.908996,11.388002,...,GGGSGIPLRPYDSYNPSVLRGPLL,PLRPYDSYNPSV,24,RPYDSY,PLRPYDSY,LRPYDSY,PYDSY,LRPYDSY,PYDSY,1
29,EDWDMLDVDEDEKLTGEEEFELLAGPLGLNDRRIVP,31,NaN,36,546,NaN,NaN,NaN,10.764586,11.127201,...,GGSGIPLREDWDMLDVDEDEKLTG,LREDWDMLDVDE,24,EDWDML,LREDWDML,REDWDML,DWDML,REDWDML,DWDML,1
37,FPPHQQHPQFNQPPHPHNFNRFPPRFMQDDSATAA,47,NaN,35,12,NaN,NaN,NaN,10.208153,11.042227,...,QFNQPPHPHNFNRFPPRFMQDDSA,HPHNFNRFPPRF,24,HNFNRF,HPHNFNRF,PHNFNRF,NFNRF,PHNFNRF,NFNRF,1
48,MNNDKHCFECYGYDIIIDES,41,NaN,20,51,NaN,NaN,NaN,10.059859,10.532193,...,NDKHCFECYGYDIIIDES------,ECYGYDIIIDES,24,YGYDII,ECYGYDII,CYGYDII,GYDII,CYGYDII,GYDII,1
49,ESTQHILNPNTSLNSNNFQILHSSHPQGNYSNSKLS,46,NaN,36,622,NaN,NaN,NaN,10.230125,10.488100,...,NPNTSLNSNNFQILHSSHPQGNYS,NSNNFQILHSSH,24,NNFQIL,NSNNFQIL,SNNFQIL,NFQIL,SNNFQIL,NFQIL,1
55,HSVVPLLDSYDLLSYDDLSH,43,NaN,20,22,NaN,NaN,NaN,10.242322,10.303895,...,PLLDSYDLLSYDDLSH--------,DLLSYDDLSH--,24,LSYDDL,DLLSYDDL,LLSYDDL,SYDDL,LLSYDDL,SYDDL,1
74,TEEQLKEKFPEADPYEIIESFNVVAKEVFVVLF,49,NaN,33,37,NaN,NaN,NaN,10.114429,9.745663,...,EADPYEIIESFNVVAKEVFVVLF-,IIESFNVVAKEV,24,ESFNVV,IIESFNVV,IESFNVV,SFNVV,IESFNVV,SFNVV,1


In [10]:
print(len(a.difference(b)))
print(len(b.difference(a)))

36
13


In [11]:
full_data_table[full_data_table['ID'].isin(screen_binders_df['ID'].to_list())]

,Unnamed: 0,ID,ER 0,ER 1,ER 3,ER 4,ER 5,ER 6,Input Count,1 Count,...,Query start,Query End,Hit Start,Hit End,E-value,Score (Bits),z_score_4,z_score_5,z_score_6,avg_z_score
0,0,DIDNFDIDDFDDDDDWEDICII,0,4.094998,9.506665,12.105609,13.229315,14.031836,976,9092,...,5.0,19.0,256.0,270.0,1.900000e+00,23.1,3.574377,4.082822,4.670163,4.11
1,1,LWQPHSSKQDDMWEHIAISML,0,4.420856,9.955033,12.104137,12.854852,13.584948,244,2849,...,1.0,17.0,465.0,481.0,4.550000e-07,42.0,3.573749,3.927170,4.482965,3.99
2,2,VIEEADLDGDGSWALLTSRT,0,6.061220,10.521183,11.960512,12.786388,12.782009,10,364,...,1.0,11.0,152.0,162.0,7.100000e-02,27.3,3.512489,3.898712,4.146620,3.85
3,3,LHNPESGEEAVALLEELQRDLGWDILA,0,5.191722,10.334677,12.020974,12.594453,12.509488,13,259,...,1.0,21.0,117.0,137.0,1.010000e-08,47.0,3.538278,3.818930,4.032463,3.80
4,4,QIPENSTSSQADDEWCYVLI,0,5.148098,10.131437,11.838327,12.629829,12.705135,91,1759,...,1.0,10.0,552.0,561.0,1.400000e+00,23.5,3.460374,3.833635,4.114419,3.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,415,MARVPLTFSDVAIASLRRSGNI,0,4.497790,6.555125,7.338737,7.601655,7.303037,19,234,...,1.0,13.0,1.0,13.0,5.000000e-03,30.4,1.541188,1.743582,1.851523,1.71
413,420,HLAAGYNNLEVAEYLLEHGADVNAQTRVV,0,4.039740,7.065926,7.296732,7.828102,7.030019,29,260,...,1.0,27.0,842.0,868.0,3.130000e-11,54.3,1.523272,1.837709,1.737157,1.70
417,424,MPNTSVVLDTDAEFVILLLFLLLLFFFFRASFA,0,4.431676,7.914457,8.140802,7.387363,6.622837,407,4788,...,1.0,15.0,642.0,656.0,1.750000e-04,35.4,1.883288,1.654507,1.566592,1.70
418,425,MSQAVQQRYSTIKQNMGTQFI,0,3.409332,7.021036,7.790198,7.932998,6.418514,48,278,...,NaN,NaN,NaN,NaN,NaN,NaN,1.733747,1.881311,1.481003,1.70


In [12]:
full_data_table[full_data_table['ID']=='DIDNFDIDDFDDDDDWEDICII']

,Unnamed: 0,ID,ER 0,ER 1,ER 3,ER 4,ER 5,ER 6,Input Count,1 Count,...,Query start,Query End,Hit Start,Hit End,E-value,Score (Bits),z_score_4,z_score_5,z_score_6,avg_z_score
0,0,DIDNFDIDDFDDDDDWEDICII,0,4.094998,9.506665,12.105609,13.229315,14.031836,976,9092,...,5.0,19.0,256.0,270.0,1.9,23.1,3.574377,4.082822,4.670163,4.11


In [14]:
screen_binders_from_raw[screen_binders_from_raw['ID']=='DIDNFDIDDFDDDDDWEDICII']

,Unnamed: 0,ID,ER 0,ER 1,ER 3,ER 4,ER 5,ER 6,Input Count,1 Count,...,Score (Bits),z_score_4,z_score_5,z_score_6,avg_z_score,regex_matches,num_regex_matches,7mer,motif_start,motif_end
0,0,DIDNFDIDDFDDDDDWEDICII,0,4.094998,9.506665,12.105609,13.229315,14.031836,976,9092,...,23.1,3.574377,4.082822,4.670163,4.11,"(DDDWEDI, 12, 18)",1,DDDWEDI,12,18


# junk